<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Cross Validation - Decision Tree - Diamonds
</b></font> </br></p>

---

# 0  | Install & Import
***

In [ ]:
# Install

In [ ]:
# Daten & Strukturen
from pandas import read_csv  # CSV laden

# Datenvorverarbeitung
from sklearn.preprocessing import Ordinalcoder  # Kategoriale Variablen encodieren

# Datenaufteilung & Cross-Validation
from sklearn.model_selection import train_test_split, KFold, cross_validate  # Split und CV

# Modell
from sklearn.tree import DecisionTreeRegressor  # Regressionsmodell

# Evaluation & Metriken
from sklearn.metrics import r2_score  # Regressionsmetriken

# Visualisierung
import plotly.express as px  # Einfache Plots
import plotly.subplots as sp  # Subplots erstellen

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1 | Understand
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>
✅ Passenden Transfer-Case als Orientierung auswählen</br>

<p><font color='black' size="5">
Anwendungsfall
</font></p>

---

Dieser klassische Datensatz enthält die Preise und andere Attribute von fast 54.000 Diamanten.



[DataSet](https://www.openml.org/search?type=data&status=active&id=42225)

[Info](https://www.kaggle.com/datasets/shivam2503/diamonds)


In [ ]:
df = read_csv(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/diamonds.csv",
    usecols=[
        "carat",
        "cut",
        "color",
        "clarity",
        "depth",
        "table",
        "price",
    ],
)

In [ ]:
data = df.copy()
target = data.pop("price")

<p><font color='black' size="5">
EDA (Exploratory Data Analysis) mit Pandas
</font></p>

In [ ]:
data.info()

In [ ]:
data.describe().T

In [ ]:
data.groupby("cut").count()

In [ ]:
data.groupby("color").count()

<p><font color='black' size="5">
EDA (Exploratory Data Analysis) mit Plotly
</font></p>

In [ ]:
title_ = "Depth"
b1 = px.box(data["depth"], title=title_, width=600, height=600)

title_ = "Carat"
b2 = px.box(data["carat"], title=title_, width=600, height=600)

title_ = "Table"
b3 = px.box(data["table"], title=title_, width=600, height=600)

fig = sp.make_subplots(rows=1, cols=3, subplot_titles=("Depth", "Carat", "Table"))

for trace in b1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in b2.data:
    fig.add_trace(trace, row=1, col=2)

for trace in b3.data:
    fig.add_trace(trace, row=1, col=3)

# Layout anpassen
fig.update_layout(width=1000, height=500, title_text="Box-Plots")

# Plot anzeigen
fig.show()

# 2 |  Prepare

---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Duplikate ermitteln/löschen</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>
✅ Preprocessing-Entscheidungen aus dem Transfer-Case auf Übertragbarkeit prüfen</br>

<p><font color='black' size="5">
Datentyp ermitteln
</font></p>

In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<p><font color='black' size="5">
Train-Test-Split
</font></p>


In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.2, random_state=42)

<p><font color='black' size="5">
Kodierung
</font></p>

In [ ]:
cat_seq = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["J", "I", "H", "G", "F", "E", "D"],
    ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],]

coder = Ordinalcoder(categories=cat_seq)
coder.fit(data_train[cat_col])
data_train[cat_col] = coder.transform(data_train[cat_col])
data_test[cat_col] = coder.transform(data_test[cat_col])

# 3 | Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Modellansatz aus dem Transfer-Case als Orientierung für die eigene Lösung nutzen</br>

<p><font color='black' size="5">🌳 Algorithmus: Entscheidungsbaum (Decision Tree – Regression)</font></p>

Ein Entscheidungsbaum für Regression funktioniert wie der Klassifikationsbaum – statt einer Klasse sagt er am Ende jedoch einen kontinuierlichen Zahlenwert vorher (z.B. einen Preis). In den Blättern steht der Mittelwert der Trainingsdaten, die in dieses Blatt fallen.

**Analogie:** Wie ein Sachverständiger, der den Wert einer Immobilie anhand von Merkmalen (Größe, Lage, Baujahr) schrittweise einschränkt.

**Wichtige Parameter:**

| Parameter | Bedeutung |
|-----------|----------|
| `max_depth` | Maximale Tiefe (begrenzt Overfitting) |
| `criterion` | Gütekriterium: `squared_error` oder `absolute_error` |
| `random_state` | Reproduzierbarkeit |

**Wann Kreuzvalidierung (K-Fold)?**

**In der Praxis relevant wenn:**
- Die Modellgüte zuverlässiger geschätzt werden soll als mit einem einzelnen Train-Test-Split
- Der Datensatz mittelgroß ist und jede Beobachtung sowohl für Training als auch Validierung genutzt werden soll
- Overfitting erkannt werden soll: ein großer Unterschied zwischen Train- und Validation-Score zeigt Overfitting

**Nicht geeignet wenn:**
- Der Datensatz sehr groß ist (> 1 Mio. Zeilen) → ein einfacher Split reicht; CV ist dann zu langsam
- Zeitreihendaten vorliegen → Standard-KFold verletzt die zeitliche Reihenfolge; dann `TimeSeriesSplit` verwenden

**Typischer Fehler:**
Cross-Validation auf dem gesamten Datensatz inklusive Preprocessing anwenden — das führt zu Data Leakage. Preprocessing (Skalierung, Encoding) muss innerhalb der CV-Schleife auf jedem Fold-Training neu gefittet werden.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Entscheidungsbaum – Regression</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
flowchart TD
    ROOT[Wurzel: Karat > 1.0?] -->|ja| A[Klarheit: SI2 oder schlechter?]
    ROOT -->|nein| B[Karat > 0.5?]

    A -->|ja| C[Ø Preis: $5.400]
    A -->|nein| D[Ø Preis: $9.200]

    B -->|ja| E[Ø Preis: $2.100]
    B -->|nein| F[Ø Preis: $750]
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=500))

<p><font size="5">
Modelauswahl
</p>

In [ ]:
model = DecisionTreeRegressor(max_depth=4, random_state=42)

<p><font size="5">
Training
</p>

In [ ]:
model.fit(data_train, target_train)

# 4 | Evaluate
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Cross-Validation</br>
✅ Bootstrapping</br>
✅ Regularisierung prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>
✅ Ergebnisse mit dem Transfer-Case vergleichen und Unterschiede erklären</br>

<p><font color='black' size="5">
Prognose
</font></p>

In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)


<p><font color='black' size="5">
Bestimmtheitsmass
</font></p>


In [ ]:
r2_train = r2_score(target_train, target_train_pred)
print(f"Modell: {model}\n -- Train --- Bestimmtheitsmass: {r2_train:5.2f}")

In [ ]:
r2_test = r2_score(target_test, target_test_pred)
print(f"Modell: {model}\n -- Test --- Bestimmtheitsmass: {r2_test:5.2f}")


<p><font color='black' size="5">
Feature Importance
</font></p>

In [ ]:
px.bar(x=model.feature_importances_, y=data.columns).update_yaxes(
    categoryorder="total ascending"
)

<p><font size="5">
Cross-Validation - K-Fold
</p>



In [ ]:
cv = KFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
cv_results = cross_validate(
    model,
    data_train,
    target_train,
    scoring="r2",
    cv=cv,
    return_train_score=True,
    return_estimator=True,
)

In [ ]:
cv_results

In [ ]:
train_result = cv_results["train_score"].mean()
val_result = cv_results["test_score"].mean()
print(f"Train {train_result:.2f} -- Validation {val_result:.2f}")

<p><font size="5">
Datensätze der einzelnen Kreuzvalidierungen
</p>

In [ ]:
cv.get_n_splits(data_train)
print(cv)

for i, (train_index, val_index) in enumerate(cv.split(data_train)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}, count: {len(train_index):,.0f}")
    print(f"  Test:  index={val_index}, count: {len(val_index):,.0f}")

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>
✅ Grenzen der Übertragbarkeit vom Transfer-Case auf den eigenen Anwendungsfall dokumentieren</br>